In [1]:
using JLD2
using Plots
using Roots
using Statistics
using ProgressMeter

# Loading and Calculating mean adaptive basin sizes from the simulation data

In [4]:
function load_A_L_values_sorted(file_path)
    #load all possible A valkues and store them in a dict with their possible L values
    a_LA_combinations = Dict{Int, Vector{Int}}()
    jldopen(file_path, "r") do file
        a_keys = collect(keys(file))

        for k in a_keys
            m = get_group_vals(k)
            if m !== nothing
                L = parse(Int, m[1])
                A = parse(Int, m[2])

                if !haskey(a_LA_combinations, A)
                    a_LA_combinations[A] = []
                end
                push!(a_LA_combinations[A], L)
            end
            sort!(a_LA_combinations[A])
        end
    end

    a_A = sort!(collect(keys(a_LA_combinations)))

    return a_A, a_LA_combinations
end

# Helper to ensure consistent naming conventions
get_group_name(L::Int, A::Int) = "L$(L)_A$(A)"
get_group_vals(s) = match(r"L(\d+)_A(\d+)", s)

"""
    save_LA_dict(filepath, L, A, data_dict)

Stores the values from `data_dict` into the file under the specific L and A group.
If a key already exists, the new data is appended to the existing array.
"""
function save_LA_dict(filepath::String, L::Int, A::Int, data_dict::Dict)
    group_name = get_group_name(L, A)
    
    # "a+" creates the file if it doesn't exist, and appends if it does
    jldopen(filepath, "a+") do file
        # Initialize group for this L and A if not present
        if !haskey(file, group_name)
            JLD2.Group(file, group_name)
        end
        
        grp = file[group_name]
        
        for (k, v) in data_dict
            key_str = String(k)
            # Append if key already exists
            if haskey(grp, key_str)
                existing_data = grp[key_str]
                new_data = vcat(existing_data, v)
                delete!(grp, key_str)
                grp[key_str] = new_data
            else
                grp[key_str] = v
            end
        end
    end
    println("Saved $(length(data_dict)) keys to $group_name in $filepath")
end

"""
    get_sample_count(filepath, L, A, target_key)

Returns the length of the array connected to `target_key` to check how many samples exist. 
Returns 0 if the file, group, or key does not exist.
"""
function get_sample_count(filepath::String, L::Int, A::Int, target_key::String)::Int
    group_name = get_group_name(L, A)
    
    if !isfile(filepath)
        return 0
    end
    
    jldopen(filepath, "r") do file
        if haskey(file, group_name) && haskey(file[group_name], target_key)
            return length(file[group_name][target_key])
        else
            return 0
        end
    end
end

"""
    load_LA_keys(filepath, L, A, keys_to_load)

Loads a specified list of keys into a Dictionary. 
If `keys_to_load` is empty, it loads all available keys for that L and A.
"""
function load_LA_keys(filepath::String, L::Int, A::Int, keys_to_load::Vector{String}=String[])
    group_name = get_group_name(L, A)
    result = Dict{String, Any}()
    
    if !isfile(filepath)
        @warn "File $filepath does not exist."
        return result
    end
    
    jldopen(filepath, "r") do file
        if !haskey(file, group_name)
            @warn "No data found for $group_name"
            return result
        end
        
        grp = file[group_name]
        target_keys = isempty(keys_to_load) ? keys(grp) : keys_to_load
        
        for k in target_keys
            if haskey(grp, k)
                result[k] = grp[k]
            else
                @warn "Key '$k' is missing in $group_name"
            end
        end
    end
    
    return result
end

load_LA_keys

# Peak data

In [ ]:
a_A, a_LA_combinations = load_A_L_values_sorted(file_path_peaks)

#load all corresponding L values

d_mean = Dict{}()
d_std = Dict{}()

for A in a_A
    d_mean[A] = Float64[]
    d_std[A] = Float64[]
    for L in a_LA_combinations[A]
        a_AdB_size = load_LA_keys(file_path_peaks, L, A, ["a_AdB_size"])["a_AdB_size"]
        #display(a_AdB_size)
        a_AdB_size = Iterators.flatten(a_AdB_size) #flatten data
        nc = A^L #normalization constant
        m, s = mean(a_AdB_size) / nc, std(a_AdB_size) / nc
        push!(d_mean[A], m)
        push!(d_std[A], s)
    end
end

#save mean and std values to a hdf5 file
for A in a_A
    data_dict = Dict(
        "L_values" => a_LA_combinations[A],
        "mean_AdB_size" => d_mean[A],
        "std_AdB_size" => d_std[A])

    #save data as JLD2 file (which is using the HDF5 format)
    jldopen(main_path*"AdB_peaks_simulations.jld2", "a") do file
        file["A=$A"] = data_dict
    end
end


# Genotype Data

In [ ]:
main_path  = String(@__DIR__) * "/data"
file_path_genotypes = main_path * "/AdB_data_thp_cluster.jld2"
a_A, a_LA_combinations = load_A_L_values_sorted(file_path_genotypes)

file_path_genotypes_save = main_path * "/AdB_genotypes_simulations.jld2"

#first create a dict to store the mean and std values for each A and coresponding L values
d_mean_genotypes = Dict{}()
d_std_genotypes = Dict{}()

nBins = 500 #set the number of bins

for A in a_A
    for L in a_LA_combinations[A]
        #first check if the data for this L and A combination already exists in the file, if it does, skip to the next one
        sample_count = get_sample_count(file_path_genotypes_save, L, A, "mean_AdB_size")
        if sample_count > 0
            println("Data for L=$L and A=$A already exists with $sample_count samples, skipping...")
            continue
        end

        ret = load_LA_keys(file_path_genotypes, L, A, ["a_AdB_size", "a_fitness"])
        ret == 0 && continue

        #calculate the mean AdB size for each fitness bin
        fitness_bins = range(0, 1, length=nBins+1)
        a_bins = [Int64[] for _ in 1:nBins]

        #group into the bins and calculate the mean AdB size for each bin
        @showprogress dt=1.0 "A=$A | L=$L" for (a_AdB_size, a_AdB_fitness) in zip(ret["a_AdB_size"], ret["a_fitness"])
            for (AdB_size, AdB_fitness) in zip(a_AdB_size, a_AdB_fitness)
                #sort into the corresponding bin
                    bin_index = findfirst(x -> x > AdB_fitness, fitness_bins) - 1
                    #bin_index = max(bin_index, 1) # Ensure it doesn't go below 1
                    push!(a_bins[bin_index], AdB_size)
            end
        end

        mean_AdB_size = [mean(bin) / A^L for bin in a_bins]
        std_AdB_size = [std(bin) / A^L for bin in a_bins]

        #save data in file
        jldopen(file_path_genotypes_save, "a") do file
            group_name = get_group_name(L, A)
            if !haskey(file, group_name)
                JLD2.Group(file, group_name)
            end
            grp = file[group_name]
            grp["fitness_bins"] = collect(fitness_bins)[1:end-1] .+ 1/(nBins*2) #bin centers  #bin centers
            grp["mean_AdB_size"] = mean_AdB_size
            grp["std_AdB_size"] = std_AdB_size
        end
    end
end

Data for L=5 and A=2 already exists with 500 samples, skipping...
Data for L=6 and A=2 already exists with 500 samples, skipping...
Data for L=7 and A=2 already exists with 500 samples, skipping...
Data for L=8 and A=2 already exists with 500 samples, skipping...
Data for L=9 and A=2 already exists with 500 samples, skipping...
Data for L=10 and A=2 already exists with 500 samples, skipping...
Data for L=11 and A=2 already exists with 500 samples, skipping...
Data for L=12 and A=2 already exists with 500 samples, skipping...
Data for L=13 and A=2 already exists with 500 samples, skipping...
Data for L=5 and A=3 already exists with 500 samples, skipping...
Data for L=6 and A=3 already exists with 500 samples, skipping...
Data for L=7 and A=3 already exists with 500 samples, skipping...
Data for L=8 and A=3 already exists with 500 samples, skipping...
Data for L=4 and A=4 already exists with 500 samples, skipping...
Data for L=5 and A=4 already exists with 500 samples, skipping...
Data f